In [3]:
import os,warnings
import joblib 
import random
import json
import datetime
import tensorflow   as tf
import numpy as np
import keras

from tensorflow.keras.optimizers    import Adam
from tensorflow.keras.callbacks     import EarlyStopping, ReduceLROnPlateau
from tqdm                           import tqdm
from preProcessingData              import PreProcessData
from utility.logger                 import aiLogger
from utility.matpilotDraw           import MatPlot
from tKanModel                          import vaeTKAN
import matplotlib.pyplot            as plt
from keras.models                   import load_model


os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # disabling gpu usage because my cuda is corrupted, needs to be fixed. 

In [5]:
with open('model/modelConfig.json', 'r') as configFile:
        config = json.load(configFile)
kanConfig = config["kan"]  

# ---------------------------------    
# Reactor, Separator,Stripper
# ---------------------------------
component  = kanConfig["component"]
modelName  = "kan"


os.environ['CUDA_VISIBLE_DEVICES'] = '1'
# strategy = tf.distribute.MirroredStrategy()


np.random.seed(1)
random.seed(1)
# ---------------------------------
# Pre processing data
# ---------------------------------
preData             = PreProcessData()
rawData             = preData.loadCriticalData()                                            # Load the data from critical file
addedVariablesData  = preData.additionEfficiencyVar(rawData)        # Add some additional efficiency variable
_,healthyPos    = preData.getHealthyState(rawData)                                      # Load the healthy state for display purpose only
clearnData      = preData.cleanData(rawData[preData.crusher1Value])                     # Clean the data by using moving average
norData         = preData.normalizeBuilinFunction(clearnData[preData.crusher1Value])    # Normalize the data


normalizedAllData           = preData.normalizeBuilinFunction(addedVariablesData,  scaler = MinMaxScaler())         # Normalize the all data
normalizedHealthy,_         = preData.getHealthyState(normalizedAllData)       # Load the healthy state
trajectory                  = preData.getTrajectory(normalizedAllData)         # Extract the trajectory

# ======================================
#  PREPROCESS THE DATA
# --------------------------------------
inputX          = norData[preData.beauEfficiencyCr1 + preData.addEfficiencyCr1]                                   # Get the clean manipulated name
seqData         = preData.splitAndPadSequences([inputX,inputX],seqLength=kanConfig['seqLength'], overlap= kanConfig['overlap']) # Split data into each sequence

print(f"The size of the data: {seqData[0].shape} and {seqData[1].shape}")




Process:143598||2024-10-02 17:54:46,833-INFO [preProcessingData.py:loadCriticalData():220]:Loading critical MOF data from files ...
Process:143598||2024-10-02 17:54:47,232-INFO [preProcessingData.py:loadCriticalData():226]:Finish loading data from files ...
Process:143598||2024-10-02 17:54:47,232-INFO [preProcessingData.py:getHealthyState():470]:Getting critical trajectories ...


KeyError: "['Eficiencia_Rendimiento', 'Z_Score_Int', 'Z_Score_Ext', 'Ratio_Flujo_Aceite', 'Rendimiento_por_Revolucion', 'Temperatura_Diferencia_Media', 'Potencia_Media', 'Eficiencia_Enfriamiento'] not in index"

In [ ]:
# ---------------------------------
# Model
# ---------------------------------
aiModel             = vaeTKAN(config=kanConfig)        

aiModel.compile(optimizer= Adam())
early_stop_loss     = 'loss'
early_stop_callback = EarlyStopping(monitor=early_stop_loss, min_delta = 1e-1, patience=5) 

history = aiModel.fit(
                        x           =   seqData,
                        y           =   seqData,
                        batch_size  =   kanConfig['batchSize'],
                        epochs      =   kanConfig['nEpoch'],
                        shuffle     =   True,
                        callbacks   =   [early_stop_callback],
                        verbose     =   1,
                        )


# ---------------------------------
# Save results
# ---------------------------------
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train","val"],loc ="upper left")

val_loss = history.history["val_loss"][-1]
plt.savefig(kanConfig['dir'] + "/" +  "loss" + str(val_loss) + ".png")